# Continue Batching

连续批处理(Continue Batching) 是一种 LLM 推理服务加速技术, 其主要解决批解码(Batching Decoding)中某些请求提前结束，导致批解码空闲问题。连续批处理是 vLLM 的关键组件技术。

在实现关键在于：

1. 主循环 continue batching inference 固定 prefill/decoding 操作
2. 求类管理 prompt 状态, 主循环中监听请求, 若有新请求则进行 prefill, 若有正在处理的 prompt 则进行 decoding
4. 增加 KV-Cache 管理：a. 初始化 batching_size, seq_len b. 状态表(使用、空闲) c. 索引(requestid, batch_id) d. 批内最大长度

伪代码为:

```python
requestor_num = 1000
reqs = Requestor(N = requestor_num)

# 与显存相关
KVengines = KVCacheEngine(batch_size, 
                          max_seq_len,)
model = model(KVengines)
Inferencer = ContinueBatchingInference(model, reqs)
next_token=torch.zeros(batch_size, 1, dtype=torch.long)

# Inferencer::Step() 主循环
while(not reqs.is_no_empty()):
  if model.kvengines.is_process():
    # do decoding
    decoding_logits = model.forward_continue_batching_decoding(next_token)
    
  if model.kvengines.is_avalable():
    prompts = reqs.get_prompts()
    if len(prompts) != 0:
      # do prefill
      prefill_logist = model.forward_conitnue_batching_prefill(prompts)
      
  # predict next token
  next_tokens = self.generate(decoding_logis, prefill_logits)
      
  # update kv-cache engines
  model.kvengines.update()
  next_tokens, req_ids = model.kvengines.rearrange(next_tokens)
  
  # reqs output
  reqs.update(req_ids, next_tokens)
```

代码实际实现逻辑与上述伪代码有一定出入

## Request

## Requestor

## KVCacheEngine

## Model

## ContinueBatchingInference

## Run